# Устройство плагина

Как реализован плагин kb_mcp: модули, доступ только через ContentsManager, хуки сохранения и события, индекс в памяти, свой MCP-транспорт, тексты инструментов в mcp_docs. Читать перед правкой кода `kb_mcp/`.

## Модули пакета kb_mcp

Пакет `kb_mcp/` в корне проекта, тесты — `tests/` (pytest на настоящем `ContentsManager` и живом Jupyter Server).

| Модуль | Роль |
|---|---|
| `__init__.py` | точка входа расширения Jupyter Server: регистрирует обработчики, хуки, автокоммит |
| `launch.py` | что раньше делали скрипты запуска: фиксированный токен (`token.txt`), вывод адресов после старта (см. `development/mcp-kb/setup#token`) |
| `store.py` | хранение: секции в Markdown-ячейках, метаданные в `metadata.kb`, хеши, id |
| `kb.py` | индекс в памяти + операции потребителя и админки |
| `hooks.py` | `pre/post_save_hook` и события `contents_service` |
| `mcp.py` | MCP-транспорт на Tornado, загрузка описаний из `mcp_docs`, `GET /mcp/docs` |
| `render.py` | плоский текст для `content` + JSON для `structuredContent` |
| `texts.py` | служебные тексты en / ru |
| `git.py` | автокоммиты Dulwich |

## Только через API сервера

Это плагин сервера, а не отдельная программа. Весь доступ к базе и к `mcp_docs` — только через `ContentsManager` (`get` / `save` / `rename` / `delete`, пути относительно корня Jupyter). Прямой доступ к файловой системе — только у Dulwich (автокоммит).

Следствия:
- работает с любым `ContentsManager`; если он не файловый (нет `root_dir`), выключается только автокоммит;
- записи MCP проходят через те же хуки, что и сохранения из веба.

## Хуки и обновление индекса

**`pre_save_hook`** — один для всех записей (веб, MCP, индексатор): ставит `id`, `hash`, `indexed`, `updated`, `changed_by`; ячейки `code`/`raw` превращает в Markdown (код — в блок ```` ``` ````). Кто пишет — из контекста вызова (`kb.WRITER`: `mcp` / `indexer`; без него — `web`). Новым ячейкам из браузера метаданные берутся из сохранённой версии по `id` ячейки Jupyter — браузер не получает их обратно после сохранения.

**Индекс в памяти** обновляется:
- `post_save_hook` — сразу после сохранения, из уже нормализованного ноутбука, без повторного чтения;
- события `contents_service` (`rename`, `delete`, `copy`, `upload`, `save`) — для новой папки создаётся шапка-заглушка;
- правки мимо Jupyter (git, редактор, adb) — перед каждым запросом MCP сверяется `last_modified`; такие секции получают `changed_by = external`.

## MCP-транспорт

Своя минимальная реализация Streamable HTTP на обработчиках Tornado Jupyter Server, **без сессий**; внешний MCP SDK не используется:
- `POST` JSON-RPC: `initialize`, `tools/list`, `tools/call`, `ping`, уведомления (ответ `202`), пакеты запросов → ответ `application/json`;
- `GET` / `DELETE` → `405`.
- Версии протокола: `2025-06-18`, `2025-03-26`, `2024-11-05` (отвечаем запрошенной, если знаем, иначе самой новой).
- Настройки подключения (`profile`, `lang`, `limit`, `off`) берутся из параметров адреса при каждом запросе.

Авторизация — токен Jupyter: `?token=` или заголовок `Authorization: token …`; XSRF-проверка для MCP отключена (клиенты — программы с токеном).

## Тексты инструментов в mcp_docs

Описания инструментов и инструкции для моделей лежат **вне базы** — в `jupyter_data/mcp_docs/<lang>/<profile>.ipynb` (`lang` = `en`/`ru`, `profile` = `lite`/`std`/`full`/`admin`; всего 8 статей). Формат — как у статьи базы:
- шапка (`id = head`) — обзор для модели: термины, вид путей, порядок работы, правила; уходит в `instructions` ответа `initialize`;
- по секции на инструмент: `id` = имя инструмента через дефис (`save-section`), заголовок = имя инструмента, текст = описание и параметры.

`admin.ipynb` — правила индексатора для `/mcp/kb-admin`: шапка — инструкция, секции — команды `unindexed`, `get-full`, `put-article`, `put-sections`. Описания команд потребителя админка берёт из `full.ipynb` того же языка.

Сервер берёт тексты для `tools/list` и `initialize` при каждом запросе (правка в Jupyter → новое описание без правки кода; кеш сбрасывается по `last_modified`). Нет файла для `lang` — берётся английский; нет и его — встроенные тексты (`ADMIN_DOCS` в `kb_mcp/mcp.py`, только для админки).

Отдаются **без авторизации**:
```
GET /mcp/docs/<lang>/<profile>        → текст шапки
GET /mcp/docs/<lang>/<profile>/<id>   → текст секции
```
Ответ — просто содержимое ячейки (`text/markdown`), без JSON.